# 31.07 - YOLO object detection fine-tuning

**Notebook type:** Solution notebook with theory, complete implementations, smoke checks, and test cases.

**Daily output:** Allowed-library one-stage detector baseline + YOLO compatibility and error report.

The current allowlist has no exact YOLO implementation. This notebook keeps the YOLO label, confidence, IoU, and one-stage detection concepts, but uses Torchvision **FCOS** as the executable competition-relevant substitute. FCOS is also a one-stage detector and is anchor-free, but it is not YOLO. The official FCOS builder supports both pretrained COCO weights and architecture-only construction.

## Core Ideas

- A YOLO detector predicts many candidates in one forward pass. Each candidate combines objectness, box coordinates, and class scores.
- A YOLO label row is `class_id x_center y_center width height`, normalized to the image dimensions. Torchvision detection models instead expect pixel `xyxy` boxes and positive class IDs, so the Dataset is the adaptation boundary.
- **FCOS is the allowed substitute, not an exact YOLO implementation.** It is a fully convolutional one-stage, anchor-free detector. Its head predicts class scores, box regression, and center-ness across feature-pyramid locations.
- `FCOS_ResNet50_FPN_Weights.DEFAULT` supplies official COCO-pretrained weights, but may trigger a download. Use it only when the checkpoint is competition-legal and cached or attached. `weights=None, weights_backbone=None` constructs the official architecture fully offline.
- Confidence filtering decides which candidates remain; IoU affects matching, NMS, and evaluation. `mAP@50` uses one IoU threshold, while `mAP@50:95` averages thresholds from 0.50 to 0.95.
- Validation images must remain untouched by training. Error reports should separate true positives, false positives, and false negatives by class.

Official model reference: https://docs.pytorch.org/vision/stable/models/fcos.html

The prepared images are deliberately tiny. The offline run below proves the data/model/training contracts; it is not a claim that randomly initialized FCOS weights are competitive. For a real competition, set the explicit pretrained option only after confirming the checkpoint rules and local availability.

In [ ]:
import os
import csv
import time
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fcos_resnet50_fpn, FCOS_ResNet50_FPN_Weights
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

SEED = 31
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_ROOT = "_day31_yolo_data"
IMAGE_DIR = os.path.join(DATA_ROOT, "images")
LABEL_DIR = os.path.join(DATA_ROOT, "labels")
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(LABEL_DIR, exist_ok=True)
CLASS_NAMES = ["warm_square", "cool_square"]

## Prepared YOLO-Style Dataset

The provided generator writes 48 images, one label text file per image, a disjoint stratified split CSV, and a human-readable `dataset.yaml`. Each class has six validation observations, which is enough for this structural tutorial but below the preferred ten-per-class target; interpret per-class metrics cautiously.

In [ ]:
manifest_rows = []
for class_id in range(2):
    for class_index in range(24):
        image = Image.new("RGB", (64, 64), color=(232, 235, 239))
        draw = ImageDraw.Draw(image)
        x1 = 7 + ((class_index * 7 + class_id * 5) % 29)
        y1 = 8 + ((class_index * 11 + class_id * 3) % 27)
        side = 17 + (class_index % 5)
        x2, y2 = x1 + side, y1 + side
        color = (220, 70, 65) if class_id == 0 else (55, 125, 225)
        draw.rectangle((x1, y1, x2, y2), fill=color, outline=(30, 30, 30), width=1)
        stem = f"class{class_id}_{class_index:02d}"
        image.save(os.path.join(IMAGE_DIR, stem + ".png"))
        xc, yc = (x1 + x2) / 128.0, (y1 + y2) / 128.0
        width, height = (x2 - x1) / 64.0, (y2 - y1) / 64.0
        with open(os.path.join(LABEL_DIR, stem + ".txt"), "w", encoding="utf-8") as handle:
            handle.write(f"{class_id} {xc:.6f} {yc:.6f} {width:.6f} {height:.6f}\n")
        split = "val" if class_index % 4 == 0 else "train"
        manifest_rows.append({"stem": stem, "split": split, "class_id": class_id})

with open(os.path.join(DATA_ROOT, "manifest.csv"), "w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["stem", "split", "class_id"])
    writer.writeheader()
    writer.writerows(manifest_rows)

for split_name in ["train", "val"]:
    with open(os.path.join(DATA_ROOT, split_name + ".txt"), "w", encoding="utf-8") as handle:
        for row in manifest_rows:
            if row["split"] == split_name:
                handle.write(os.path.join("images", row["stem"] + ".png") + "\n")

with open(os.path.join(DATA_ROOT, "dataset.yaml"), "w", encoding="utf-8") as handle:
    handle.write("path: _day31_yolo_data\ntrain: train.txt\nval: val.txt\nnc: 2\nnames: [warm_square, cool_square]\n")

train_rows = [row for row in manifest_rows if row["split"] == "train"]
val_rows = [row for row in manifest_rows if row["split"] == "val"]
print("train/validation sizes:", len(train_rows), len(val_rows))
print("validation support:", np.bincount([row["class_id"] for row in val_rows], minlength=2).tolist())

## Exercise 31-A: Adapt YOLO files to Torchvision detection targets

Load each image as `[3,64,64]` float data in `[0,1]`. Convert the single normalized YOLO `xywh` row to pixel `xyxy`, and shift class IDs from `{0,1}` to Torchvision's positive foreground IDs `{1,2}`. Dataset rows are supplied explicitly so the split cannot leak.

**Return structure — `YoloToTorchvisionDataset`:** A `torch.utils.data.Dataset` of length `N`. `dataset[i]` returns a dictionary with `image`: CPU `torch.float32 [3,64,64]`; `target`: a dictionary containing `boxes`, CPU `torch.float32 [1,4]` pixel `xyxy`, and `labels`, CPU `torch.int64 [1]` with values in `{1,2}`; `class_id`: scalar CPU `torch.int64` in `{0,1}` for reporting; and `stem`: Python `str`.

In [ ]:
class YoloToTorchvisionDataset(Dataset):
    def __init__(self, rows, image_dir=IMAGE_DIR, label_dir=LABEL_DIR):
        self.rows = list(rows)
        self.image_dir = image_dir
        self.label_dir = label_dir

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows[index]
        stem = row["stem"]
        image_array = np.asarray(
            Image.open(os.path.join(self.image_dir, stem + ".png")).convert("RGB"),
            dtype=np.float32,
        ) / 255.0
        image = torch.from_numpy(image_array.transpose(2, 0, 1).copy())
        with open(os.path.join(self.label_dir, stem + ".txt"), "r", encoding="utf-8") as handle:
            parts = handle.readline().strip().split()
        if len(parts) != 5:
            raise ValueError(f"Expected one five-value YOLO row for {stem}")
        class_id = int(parts[0])
        xc, yc, width, height = [float(value) for value in parts[1:]]
        if class_id not in (0, 1) or any(value < 0.0 or value > 1.0 for value in [xc, yc, width, height]):
            raise ValueError(f"Invalid YOLO target for {stem}")
        image_height, image_width = image.shape[1:]
        box = torch.tensor([[
            (xc - width / 2.0) * image_width,
            (yc - height / 2.0) * image_height,
            (xc + width / 2.0) * image_width,
            (yc + height / 2.0) * image_height,
        ]], dtype=torch.float32)
        target = {
            "boxes": box,
            "labels": torch.tensor([class_id + 1], dtype=torch.int64),
        }
        return {
            "image": image,
            "target": target,
            "class_id": torch.tensor(class_id, dtype=torch.int64),
            "stem": stem,
        }


# Smoke check: load and convert one YOLO observation.
train_dataset = YoloToTorchvisionDataset(train_rows)
val_dataset = YoloToTorchvisionDataset(val_rows)
sample = train_dataset[0]
print(sample["stem"], sample["image"].shape, sample["target"])

## Exercise 31-B: Build an allowed one-stage detector

Use `torchvision.models.detection.fcos_resnet50_fpn` instead of hand-coding a detector. The offline branch must set both `weights=None` and `weights_backbone=None`. The pretrained branch loads official COCO weights and replaces only the classification logits layer for the custom class count; it can download a checkpoint and should therefore be opt-in.

Torchvision detection models reserve label `0` for background, so two foreground classes require `num_classes=3`.

**Return structure — `build_allowed_fcos`:** A callable `torchvision.models.detection.FCOS` module on CPU. In training mode it accepts `list[torch.float32 Tensor[3,H,W]]` plus a same-length `list[dict]` of `boxes` and `labels`, and returns a dictionary of scalar loss tensors. In evaluation mode it accepts the image list and returns `list[dict]`; every dictionary has `boxes` (`float32 [K,4]`), `scores` (`float32 [K]`), and `labels` (`int64 [K]`) on the model device. The function itself performs no forward pass.

In [ ]:
def build_allowed_fcos(num_classes=3, use_pretrained=False, min_size=64, max_size=64):
    if use_pretrained:
        model = fcos_resnet50_fpn(
            weights=FCOS_ResNet50_FPN_Weights.DEFAULT,
            min_size=min_size,
            max_size=max_size,
        )
        old_logits = model.head.classification_head.cls_logits
        num_anchors = model.anchor_generator.num_anchors_per_location()[0]
        new_logits = nn.Conv2d(
            old_logits.in_channels,
            num_anchors * num_classes,
            kernel_size=3,
            stride=1,
            padding=1,
        )
        torch.nn.init.normal_(new_logits.weight, std=0.01)
        torch.nn.init.constant_(new_logits.bias, -float(np.log((1.0 - 0.01) / 0.01)))
        model.head.classification_head.cls_logits = new_logits
    else:
        model = fcos_resnet50_fpn(
            weights=None,
            weights_backbone=None,
            num_classes=num_classes,
            min_size=min_size,
            max_size=max_size,
        )
    return model.cpu()


# Smoke check: architecture-only construction is deterministic and offline.
fcos_smoke_model = build_allowed_fcos(num_classes=3, use_pretrained=False)
print(type(fcos_smoke_model).__name__, "backbone channels:", fcos_smoke_model.backbone.out_channels)

## Exercise 31-C: Run one native FCOS training step

Let the official model calculate its classification, box-regression, and center-ness losses. This is the correct library boundary: the learner assembles images and target dictionaries rather than recreating FCOS loss internals.

**Return structure — `detection_training_step`:** A dictionary with Python `float` values `classification`, `bbox_regression`, `bbox_ctrness`, and `total`. The function performs one optimizer update on `model` using every item in `batch`, a non-empty `list[dict]` following the Dataset schema, and leaves the model in training mode.

In [ ]:
def detection_training_step(model, batch, optimizer, device=DEVICE):
    model.to(device)
    model.train()
    images = [item["image"].to(device) for item in batch]
    targets = [
        {"boxes": item["target"]["boxes"].to(device), "labels": item["target"]["labels"].to(device)}
        for item in batch
    ]
    optimizer.zero_grad()
    losses = model(images, targets)
    total = sum(losses.values())
    total.backward()
    optimizer.step()
    result = {key: float(value.detach().cpu()) for key, value in losses.items()}
    result["total"] = float(total.detach().cpu())
    return result


# Smoke check: one official FCOS optimization step on two prepared observations.
fcos_smoke_optimizer = torch.optim.SGD(fcos_smoke_model.parameters(), lr=0.0005, momentum=0.9)
smoke_losses = detection_training_step(fcos_smoke_model, [train_dataset[0], train_dataset[1]], fcos_smoke_optimizer)
print("FCOS native losses:", smoke_losses)

## Exercise 31-D: Run a controlled full-split FCOS baseline

Train on every prepared training observation for one structural epoch. `use_pretrained=False` keeps this notebook offline and tests the official architecture; `True` is the competition-oriented transfer-learning route but requires a permitted, available COCO checkpoint. Do not compare accuracy between these routes unless initialization is the intended variable and the full experiment is controlled.

**Return structure — `fit_detector`:** A tuple `(model, history)`. Position 0 is a trained Torchvision `FCOS` module on `device`. Position 1 is a list of exactly `epochs` dictionaries; each has Python `int` `epoch`, Python `int` `observations`, and Python `float` values `classification`, `bbox_regression`, `bbox_ctrness`, and `total`. The function prints train size, initialization route, observations per epoch, and runtime.

In [ ]:
def fit_detector(dataset, epochs=1, batch_size=4, learning_rate=0.0005, use_pretrained=False, device=DEVICE):
    torch.manual_seed(SEED)
    model = build_allowed_fcos(num_classes=3, use_pretrained=use_pretrained).to(device)
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
        collate_fn=lambda items: items,
    )
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
    history = []
    start = time.perf_counter()
    for epoch in range(1, epochs + 1):
        totals = {"classification": 0.0, "bbox_regression": 0.0, "bbox_ctrness": 0.0, "total": 0.0}
        observations = 0
        for batch in loader:
            losses = detection_training_step(model, batch, optimizer, device)
            count = len(batch)
            observations += count
            for key in totals:
                totals[key] += losses[key] * count
        history.append({
            "epoch": epoch,
            "observations": observations,
            **{key: totals[key] / observations for key in totals},
        })
    runtime = time.perf_counter() - start
    print("train size:", len(dataset), "pretrained COCO:", use_pretrained)
    print("epochs:", epochs, "observations/epoch:", len(dataset), "runtime seconds:", round(runtime, 3))
    return model, history


# Smoke check: full prepared training split with the offline official architecture.
trained_detector, training_history = fit_detector(train_dataset, epochs=1, batch_size=4, use_pretrained=False)
print("training evidence:", training_history[-1])

## Exercise 31-E: Categorize detector errors

Run the native FCOS evaluation output through a one-object-per-image matcher. Keep only the highest-confidence prediction for this compact fixture. A real detector evaluation should retain all candidates and calculate AP/mAP with a standard evaluator.

**Return structure — `evaluate_detector`:** A dictionary with `num_images`: Python `int`; `confidence_threshold` and `iou_threshold`: Python `float`; `support`, `tp`, `fp`, and `fn`: Python lists of two integers; `precision`, `recall`, and `f1`: Python lists of two floats; `macro_f1`: Python `float`; and `rows`: a `list[dict]` of length `num_images`. Each row has `stem` (`str`), `truth` (`int`, zero-based report class), `predicted` (`int` zero-based or `-1`), `confidence` (`float`), `iou` (`float`), and `outcome` (`TP`, `FP+FN`, or `FN`).

In [ ]:
def evaluate_detector(model, dataset, confidence_threshold=0.2, iou_threshold=0.5, device=DEVICE):
    model.to(device)
    model.eval()
    support = [0, 0]
    tp = [0, 0]
    fp = [0, 0]
    fn = [0, 0]
    rows = []
    with torch.no_grad():
        for item in dataset:
            output = model([item["image"].to(device)])[0]
            truth = int(item["class_id"])
            support[truth] += 1
            if len(output["scores"]) == 0 or float(output["scores"][0]) < confidence_threshold:
                predicted = -1
                confidence = 0.0 if len(output["scores"]) == 0 else float(output["scores"][0])
                iou = 0.0
                outcome = "FN"
                fn[truth] += 1
            else:
                predicted = int(output["labels"][0]) - 1
                confidence = float(output["scores"][0])
                predicted_box = output["boxes"][0].cpu()
                target_box = item["target"]["boxes"][0]
                top_left = torch.maximum(predicted_box[:2], target_box[:2])
                bottom_right = torch.minimum(predicted_box[2:], target_box[2:])
                intersection_wh = (bottom_right - top_left).clamp(min=0)
                intersection = float(intersection_wh[0] * intersection_wh[1])
                predicted_area = float((predicted_box[2] - predicted_box[0]).clamp(min=0) * (predicted_box[3] - predicted_box[1]).clamp(min=0))
                target_area = float((target_box[2] - target_box[0]) * (target_box[3] - target_box[1]))
                iou = intersection / max(predicted_area + target_area - intersection, 1e-7)
                if predicted == truth and iou >= iou_threshold:
                    outcome = "TP"
                    tp[truth] += 1
                else:
                    outcome = "FP+FN"
                    if predicted in (0, 1):
                        fp[predicted] += 1
                    fn[truth] += 1
            rows.append({
                "stem": item["stem"], "truth": truth, "predicted": predicted,
                "confidence": confidence, "iou": float(iou), "outcome": outcome,
            })
    precision = [tp[c] / max(tp[c] + fp[c], 1) for c in range(2)]
    recall = [tp[c] / max(tp[c] + fn[c], 1) for c in range(2)]
    f1 = [2 * precision[c] * recall[c] / max(precision[c] + recall[c], 1e-12) for c in range(2)]
    return {
        "num_images": len(dataset),
        "confidence_threshold": float(confidence_threshold),
        "iou_threshold": float(iou_threshold),
        "support": support, "tp": tp, "fp": fp, "fn": fn,
        "precision": precision, "recall": recall, "f1": f1,
        "macro_f1": float(np.mean(f1)), "rows": rows,
    }


# Smoke check: full untouched validation split and interpretable evidence.
detection_report = evaluate_detector(trained_detector, val_dataset)
print("validation images:", detection_report["num_images"])
print("validation support:", detection_report["support"])
print("per-class precision:", detection_report["precision"])
print("per-class recall:", detection_report["recall"])
print("per-class F1:", detection_report["f1"])
print("Macro-F1:", detection_report["macro_f1"])
print("outcome counts:", {name: sum(row["outcome"] == name for row in detection_report["rows"]) for name in ["TP", "FP+FN", "FN"]})

## Test Cases

The tests enforce contracts and split integrity, not a stochastic claim that this tiny model must achieve a specific accuracy.

**Return structure — `run_day31_tests`:** Returns `None`. Success is communicated by assertions completing and the exact printed message `Day 31 tests passed`.

In [ ]:
def run_day31_tests():
    assert os.path.isfile(os.path.join(DATA_ROOT, "dataset.yaml"))
    assert os.path.isfile(os.path.join(DATA_ROOT, "manifest.csv"))
    assert os.path.isfile(os.path.join(DATA_ROOT, "train.txt"))
    assert os.path.isfile(os.path.join(DATA_ROOT, "val.txt"))
    train_stems = {row["stem"] for row in train_rows}
    val_stems = {row["stem"] for row in val_rows}
    assert train_stems.isdisjoint(val_stems)
    assert len(train_stems | val_stems) == 48
    assert np.bincount([row["class_id"] for row in val_rows], minlength=2).tolist() == [6, 6]
    item = train_dataset[0]
    assert set(item) == {"image", "target", "class_id", "stem"}
    assert item["image"].shape == (3, 64, 64) and item["image"].dtype == torch.float32
    assert set(item["target"]) == {"boxes", "labels"}
    assert item["target"]["boxes"].shape == (1, 4) and item["target"]["boxes"].dtype == torch.float32
    assert item["target"]["labels"].shape == (1,) and item["target"]["labels"].dtype == torch.int64
    assert int(item["target"]["labels"][0]) in (1, 2)
    assert type(trained_detector).__name__ == "FCOS"
    assert len(training_history) == 1
    expected_history = {"epoch", "observations", "classification", "bbox_regression", "bbox_ctrness", "total"}
    assert set(training_history[0]) == expected_history and training_history[0]["observations"] == len(train_dataset)
    trained_detector.eval()
    with torch.no_grad():
        output = trained_detector([item["image"].to(DEVICE)])[0]
    assert set(output) == {"boxes", "scores", "labels"}
    assert output["boxes"].ndim == 2 and output["boxes"].shape[1] == 4
    assert output["scores"].dtype == torch.float32 and output["labels"].dtype == torch.int64
    assert detection_report["num_images"] == len(val_dataset)
    assert sum(detection_report["support"]) == len(val_dataset)
    assert len(detection_report["rows"]) == len(val_dataset)
    assert set(detection_report["rows"][0]) == {"stem", "truth", "predicted", "confidence", "iou", "outcome"}
    print("Day 31 tests passed")


run_day31_tests()

## Day 31 Checklist

- [ ] I verified that no exact YOLO implementation exists in the current allowed libraries.
- [ ] I can explain why FCOS is a useful one-stage, anchor-free substitute but is not YOLO.
- [ ] I converted normalized YOLO labels to Torchvision pixel-box targets with positive class IDs.
- [ ] I used the official FCOS builder and its native loss dictionary instead of reimplementing the detector.
- [ ] I know that `DEFAULT` weights may download and must be cached/attached and competition-legal.
- [ ] I kept validation disjoint and reported its support plus TP, FP, FN, and per-class metrics.